In [0]:
%pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128
%pip install 'mostlyai[local]==6.1.1'
%restart_python

In [0]:
# Allocator config must be set BEFORE torch initializes CUDA (post-large-allocation fragmentation).
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Fail LOUDLY if the GPU is not visible to torch (else DP training silently runs on CPU for
# hours). To intentionally train on CPU (e.g. reference-only), set widget allow_cpu=true.
dbutils.widgets.dropdown("allow_cpu", "false", ["false", "true"])
import torch
gpu_ok = torch.cuda.is_available()
print(f"torch {torch.__version__} | cuda available: {gpu_ok} | "
      f"device count: {torch.cuda.device_count() if gpu_ok else 0}")
if gpu_ok:
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, "| total VRAM GB:", round(p.total_memory / 1e9, 1))
elif dbutils.widgets.get("allow_cpu") != "true":
    raise RuntimeError(
        "torch.cuda.is_available() is False — GPU not usable (driver/torch CUDA mismatch). "
        "Fix the cu1XX index URL in the install cell to match `nvidia-smi`, or set widget "
        "allow_cpu=true to train on CPU anyway (slow, especially with DP).")

In [0]:
%run ./engine

In [0]:
%run ./omop_config

In [0]:
# ============================================================================
# The active dataset config. In-cell asserts fire BEFORE any training cost.
# ============================================================================
cfg = OMOP_CONFIG
PROFILER = cfg.profiler
RARE = cfg.rare
VALPROT_OVERRIDE = cfg.valprot_override

SRC = f"{cfg.source['catalog']}.{cfg.source['schema']}"
TGT = f"{cfg.target['catalog']}.{cfg.target['schema']}"
VOLUME = cfg.target["volume"]
GEN_DIR = f"{VOLUME}/generators"
GEN_REFERENCE = f"{GEN_DIR}/reference.zip"
GEN_CORE = f"{GEN_DIR}/core.zip"
CONFIG_DIR = f"{VOLUME}/config"
REPORT_DIR = f"{VOLUME}/reports"
MANIFEST_PATH = f"{CONFIG_DIR}/run_manifest.json"
COHORT = cfg.extra["cohort"]
OUTPUT = cfg.extra["output"]
DP = cfg.dp
MODEL_LIMITS = cfg.model_limits          # same object the engine reads — widget mutation propagates
MAX_PANDAS_ROWS = cfg.extra["max_pandas_rows"]
GRAPH = cfg.tables
print(f"cfg-first: SRC={SRC} TGT={TGT} GEN_CORE={GEN_CORE} DP_enabled={DP['enabled']}")
assert SRC == "4_prod.omop" and TGT == "8_dev.synth_omop", "cfg catalog drift — STOP"
assert GEN_CORE.endswith("/core.zip") and GEN_REFERENCE.endswith("/reference.zip"), "zip name drift — STOP"
assert GEN_CORE != GEN_REFERENCE, "gen path drift — STOP"
assert CONFIG_DIR.endswith("/config") and REPORT_DIR.endswith("/reports"), "dir drift — STOP"
assert DP["enabled"] is True and DP["delta"] == 1e-6, "DP config drift — STOP"
assert int(COHORT["persons"]) > 0 and int(OUTPUT["persons"]) > 0 and int(MAX_PANDAS_ROWS) > 0, "size config missing — STOP"

In [0]:
import time, json, shutil, glob, traceback
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import types as T
from mostlyai.sdk import MostlyAI

dbutils.widgets.dropdown("skip_reference", "false", ["false", "true"])
dbutils.widgets.dropdown("skip_train", "false", ["false", "true"])
dbutils.widgets.dropdown("skip_generate", "false", ["false", "true"])
dbutils.widgets.dropdown("quick_smoke", "false", ["false", "true"])
dbutils.widgets.text("max_training_time", str(MODEL_LIMITS["max_training_time"]))
dbutils.widgets.text("max_epochs", str(MODEL_LIMITS["max_epochs"]))
dbutils.widgets.text("local_dir", "/local_disk0/mostlyai")
dbutils.widgets.dropdown("force_ignore_stale", "false", ["false", "true"])
dbutils.widgets.dropdown("force_ignore_manifest", "false", ["false", "true"])
dbutils.widgets.text("output_persons", str(OUTPUT["persons"]))
dbutils.widgets.text("ref_pool", str(COHORT["ref_pool"]))
dbutils.widgets.dropdown("allow_large", "false", ["false", "true"])
dbutils.widgets.text("allow_incomplete", "false")

SKIP_REFERENCE = dbutils.widgets.get("skip_reference") == "true"
SKIP_TRAIN = dbutils.widgets.get("skip_train") == "true"
SKIP_GENERATE = dbutils.widgets.get("skip_generate") == "true"
QUICK = dbutils.widgets.get("quick_smoke") == "true"
if QUICK:
    MODEL_LIMITS["max_training_time"] = 3.0
    MODEL_LIMITS["max_epochs"] = 5.0
    print("QUICK_SMOKE: 3min/table smoke train. core.zip will be BACKED UP first; "
          "generation is DISABLED for this run (smoke artefacts are never generated from).")
else:
    MODEL_LIMITS["max_training_time"] = float(dbutils.widgets.get("max_training_time"))
    MODEL_LIMITS["max_epochs"] = float(dbutils.widgets.get("max_epochs"))
LOCAL_DIR = dbutils.widgets.get("local_dir").strip()
output_persons = int(dbutils.widgets.get("output_persons"))
ref_pool = int(dbutils.widgets.get("ref_pool"))
allow_large = dbutils.widgets.get("allow_large") == "true"
allow_incomplete = dbutils.widgets.get("allow_incomplete").strip().lower() == "true"
force_ignore_manifest = dbutils.widgets.get("force_ignore_manifest") == "true"
print(f"skip_reference={SKIP_REFERENCE} skip_train={SKIP_TRAIN} skip_generate={SKIP_GENERATE} "
      f"quick_smoke={QUICK} MODEL_LIMITS={MODEL_LIMITS} local_dir={LOCAL_DIR} "
      f"output_persons={output_persons} ref_pool={ref_pool} allow_large={allow_large} "
      f"allow_incomplete={allow_incomplete}")

def _utcnow():
    return datetime.now(timezone.utc).isoformat()

def read_manifest():
    try:
        with open(MANIFEST_PATH) as f:
            return json.load(f)
    except Exception:
        return None

def write_manifest(m):
    os.makedirs(CONFIG_DIR, exist_ok=True)
    with open(MANIFEST_PATH, "w") as f:
        json.dump(m, f, indent=2, default=str)

def _zip_fingerprint(path):
    try:
        st = os.stat(path)
        return {"size_bytes": int(st.st_size), "mtime": int(st.st_mtime)}
    except Exception:
        return None

# R1: require the run manifest from pipeline stage=pre (the cohort this training reads).
run_manifest = read_manifest()
if not run_manifest:
    if force_ignore_manifest:
        run_manifest = {"run_id": "legacy_unmanifested", "stage": "cohort_ready"}
        print("WARNING force_ignore_manifest=true — no run manifest; provenance NOT tracked.")
    else:
        raise RuntimeError(
            "No run_manifest.json — run `pipeline` (stage=pre) first so this training is tied "
            "to a run_id, or set force_ignore_manifest=true for a legacy cohort.")
RUN_ID = run_manifest.get("run_id", "unknown")
print(f"run_id = {RUN_ID}")

# Resource snapshot helper (disk + host RAM), captured around core training.
def resource_snapshot(tag, local_dir):
    snap = {"tag": tag}
    try:
        du = shutil.disk_usage(local_dir if os.path.exists(local_dir) else "/local_disk0")
        snap["disk_total_gb"] = round(du.total / 1e9, 1)
        snap["disk_used_gb"] = round(du.used / 1e9, 1)
        snap["disk_free_gb"] = round(du.free / 1e9, 1)
    except Exception as e:
        snap["disk_error"] = str(e)
    try:
        snap["local_dir_size_gb"] = round(sum(
            os.path.getsize(os.path.join(r, f))
            for r, _, fs in os.walk(local_dir) for f in fs
            if os.path.exists(os.path.join(r, f))) / 1e9, 2) if os.path.exists(local_dir) else 0.0
    except Exception as e:
        snap["local_dir_size_error"] = str(e)
    try:
        with open("/proc/meminfo") as f:
            mem = {l.split(":")[0]: l.split(":")[1].strip() for l in f}
        snap["mem_total"] = mem.get("MemTotal")
        snap["mem_available"] = mem.get("MemAvailable")
    except Exception as e:
        snap["mem_error"] = str(e)
    return snap

def _val(x):
    try:
        return x.value if hasattr(x, "value") else (
            x if isinstance(x, (str, int, float, bool, type(None))) else str(x))
    except Exception:
        return str(x)

In [0]:
# =====================================================================================
# GENERATION MEMORY GATE (R3) — estimate the largest generated table from the cohort's
# per-subject rate scaled to output_persons; abort if it would exceed MAX_PANDAS_ROWS
# (both generation .data() and assembly materialize full pandas). Override: allow_large.
# =====================================================================================
def generation_memory_gate():
    n_subj_cohort = spark.table(f"{TGT}.cohort_{cfg.subject}").count()
    if n_subj_cohort == 0:
        raise RuntimeError("cohort subject table is empty — run pipeline stage=pre first.")
    worst = None
    ests = {}
    for t in topo_order(cfg, "core"):
        try:
            n = spark.table(f"{TGT}.cohort_{t}").count()
        except Exception:
            n = 0
        est = int(round(n / n_subj_cohort * output_persons))
        ests[t] = est
        if worst is None or est > ests[worst]:
            worst = t
    print(f"generation size estimate (cohort rate x output_persons={output_persons}): "
          f"largest={worst}~{ests[worst]:,} rows; MAX_PANDAS_ROWS={MAX_PANDAS_ROWS:,}")
    if ests[worst] > MAX_PANDAS_ROWS and not allow_large:
        raise RuntimeError(
            f"GENERATION MEMORY GATE — estimated {worst} ~{ests[worst]:,} rows exceeds "
            f"MAX_PANDAS_ROWS={MAX_PANDAS_ROWS:,} at output_persons={output_persons}. "
            f"Generation + assembly materialize full pandas/python objects on the driver. "
            f"FIX: lower output_persons, or raise cfg MAX_PANDAS_ROWS on a bigger-RAM driver, "
            f"or set allow_large=true to proceed anyway.")
    return ests

In [0]:
# =====================================================================================
# §1 TRAIN REFERENCE — small; CPU-tolerable.
# =====================================================================================
def train_reference(mostly, sdtypes):
    frames = prepare_frames("reference")
    for t, df in frames.items():
        print(f"{t}: {df.shape}")
    tables = build_generator_tables("reference", sdtypes, frames)
    print("Reference tables:", [t["name"] for t in tables])

    t0 = time.time()
    g = mostly.train(config={"name": f"{cfg.name}_reference", "tables": tables}, start=False)
    g.training.start()
    g.training.wait(progress_bar=True)
    print(f"Reference training done in {time.time()-t0:.0f}s")

    os.makedirs(GEN_DIR, exist_ok=True)
    g.export_to_file(GEN_REFERENCE)
    print("Exported:", GEN_REFERENCE)
    try:
        g.reports(file_path=f"{REPORT_DIR}/reference_qa", display=False)
        print("QA report written to", f"{REPORT_DIR}/reference_qa")
    except Exception as e:
        print("QA report skipped:", e)

In [0]:
# =====================================================================================
# §2 TRAIN CORE — heavy DP training; C1 policy; guarded export. Returns a train record.
# =====================================================================================
def train_core(mostly, sdtypes):
    frames = prepare_frames("core")
    total_rows = sum(len(df) for df in frames.values())
    print(f"Total core training rows: {total_rows:,}")
    for t in topo_order(cfg, "core"):
        print(f"  {t}: {frames[t].shape}")
    assert total_rows <= MAX_PANDAS_ROWS, (
        f"core training rows {total_rows:,} exceed MAX_PANDAS_ROWS {MAX_PANDAS_ROWS:,}; "
        "lower cohort_persons/meas_cap in the pipeline (stage=pre) and re-run.")

    tables = build_generator_tables("core", sdtypes, frames)
    print("Core tables:", [t["name"] for t in tables])
    for t in tables:
        tmc = t.get("tabular_model_configuration", {})
        assert tmc.get("max_training_time") == MODEL_LIMITS["max_training_time"], \
            f"{t['name']} missing max_training_time cap"

    # --- C1 COLLAPSE POLICY (data-driven, freshness-guarded) ---
    with open(f"{CONFIG_DIR}/rarity_profile.json") as f:
        _profile = json.load(f)
    _cohort_mode = _profile.get("header", {}).get("cohort_mode")
    if not _cohort_mode:
        _cohort_mode = "fixture" if COHORT["persons"] <= 200 else "full"
    _fresh = profile_is_fresh(_profile, RARITY_CONFIG_HASH(), _cohort_mode)
    if not _fresh:
        msg = (f"rarity_profile.json STALE/mismatch — header={_profile.get('header')}, "
               f"expected config_hash={RARITY_CONFIG_HASH()} cohort_mode={_cohort_mode}. "
               f"Re-run pipeline stage=pre (it writes a fresh profile).")
        if dbutils.widgets.get("force_ignore_stale") == "true":
            print("WARNING force_ignore_stale=true: " + msg)
        else:
            raise RuntimeError(msg)

    _policies = resolve_all_policies(topo_order(cfg, "core"), _profile.get("tables", {}),
                                     VALPROT_OVERRIDE, legacy_valprot_off(), RARE["seqlen_support_min"])
    for tcfg in tables:
        pol = _policies[tcfg["name"]]
        tmc = tcfg.setdefault("tabular_model_configuration", {})
        if pol["value_protection"]:
            tmc.pop("value_protection", None)
        else:
            tmc["value_protection"] = False
    print("resolved value_protection:", {t: _policies[t]["value_protection"] for t in topo_order(cfg, "core")})
    with open(f"{REPORT_DIR}/resolved_policy.json", "w") as f:
        json.dump({"config_hash": RARITY_CONFIG_HASH(), "cohort_mode": _cohort_mode, "fresh": _fresh,
                   "seqlen_support_min": RARE["seqlen_support_min"], "override": VALPROT_OVERRIDE,
                   "policies": _policies}, f, indent=2)

    # --- DP boundary + HONEST composed epsilon ---
    dp_tables = [t["name"] for t in tables
                 if "differential_privacy" in t.get("tabular_model_configuration", {})]
    assert (not DP["enabled"]) or len(dp_tables) == len(tables), "DP must be on EVERY core table"
    vp_off = [t["name"] for t in tables
              if t.get("tabular_model_configuration", {}).get("value_protection") is False]
    vp_on = [t["name"] for t in tables if t["name"] not in vp_off]
    _resolved_off = sorted(t for t in topo_order(cfg, "core") if not _policies[t]["value_protection"])
    assert sorted(vp_off) == _resolved_off, (
        f"applied vp_off {sorted(vp_off)} != resolved policy {_resolved_off}")
    ce = composed_epsilon(vp_on)
    print(f"DP tables: {len(dp_tables)}/{len(tables)} | value_protection ON: {len(vp_on)}/{len(tables)} "
          f"(OFF: {sorted(vp_off)}) | composed eps={ce['total']}")

    # --- TRAIN (bounded, diagnosable) ---
    snap_before = resource_snapshot("before_train", LOCAL_DIR)
    print("resources before:", json.dumps(snap_before))
    t0 = time.time()
    g = mostly.train(config={"name": f"{cfg.name}_core", "tables": tables}, start=False)
    g.training.start()
    g.training.wait(progress_bar=True)
    print(f"Core training wait() returned in {time.time()-t0:.0f}s")

    g_reloaded = mostly.generators.get(g.id)
    overall = _val(getattr(g_reloaded, "training_status", None))
    print("generator.training_status =", overall)
    snap_after = resource_snapshot("after_train", LOCAL_DIR)
    os.makedirs(REPORT_DIR, exist_ok=True)

    logs_note = None
    try:
        pth = g.training.logs(file_path=f"{REPORT_DIR}/core_train_logs.zip")
        logs_note = f"g.training.logs -> {pth}"
    except Exception as e1:
        try:
            found = (glob.glob(f"{LOCAL_DIR}/**/*.log", recursive=True)
                     + glob.glob(f"{LOCAL_DIR}/**/*progress*", recursive=True))
            copied = 0
            for src_f in found[:80]:
                try:
                    shutil.copyfile(src_f, f"{REPORT_DIR}/core_train_logs__" + src_f.replace("/", "_")[-180:]); copied += 1
                except Exception:
                    pass
            logs_note = f"g.training.logs failed ({e1}); copied {copied} files from local_dir"
        except Exception as e2:
            logs_note = f"log capture failed: logs()={e1}; local_dir copy={e2}"
    print("log capture:", logs_note)

    steps = []
    try:
        prog = g_reloaded.training.progress()
        for s in (getattr(prog, "steps", None) or []):
            msg = _val(getattr(s, "messages", "")) or ""
            steps.append({"model": _val(getattr(s, "model_label", getattr(s, "name", "?"))),
                          "step": _val(getattr(s, "step_code", getattr(s, "step", "?"))),
                          "status": _val(getattr(s, "status", "?")),
                          "messages": msg[:500]})
    except Exception as e:
        steps.append({"progress_introspection_error": str(e)})

    failed_steps = [s for s in steps if str(s.get("status", "")).upper() not in ("DONE", "?", "")]
    first_failed = next((s["model"] for s in steps if str(s.get("status", "")).upper() == "FAILED"), None)
    train_status = {"generator_id": str(g.id), "training_status": overall, "quick_smoke": QUICK,
                    "wait_seconds": round(time.time() - t0, 1), "local_dir": LOCAL_DIR,
                    "first_failed_model": first_failed, "resources_before": snap_before,
                    "resources_after": snap_after, "log_capture": logs_note,
                    "n_steps": len(steps), "failed_or_nondone_steps": failed_steps, "all_steps": steps}
    with open(f"{REPORT_DIR}/train_status.json", "w") as f:
        json.dump(train_status, f, indent=2)
    print("train_status.json written. status =", overall, "| first failed table:", first_failed)
    if failed_steps:
        print("NON-DONE STEPS:", json.dumps(failed_steps, indent=2)[:3000])
    if str(overall).upper() != "DONE":
        raise RuntimeError(
            f"Core training did NOT finish DONE (status={overall}, first failed table={first_failed}). "
            f"Not exporting a partial generator. See {REPORT_DIR}/train_status.json + core_train_logs.zip.")

    # --- quick_smoke: back up the existing GOOD core.zip before overwriting with the smoke artefact ---
    if QUICK and os.path.exists(GEN_CORE):
        backup = GEN_CORE + ".pre_quick_smoke_backup"
        shutil.copyfile(GEN_CORE, backup)
        print(f"quick_smoke: backed up existing core.zip -> {backup}")

    os.makedirs(GEN_DIR, exist_ok=True)
    g.export_to_file(GEN_CORE)
    print("Exported:", GEN_CORE)
    if QUICK:
        print("NOTE: quick_smoke model exported — SMOKE artefact (under-trained). Generation is "
              "DISABLED this run; restore the backup before any real generation.")
    try:
        g.reports(file_path=f"{REPORT_DIR}/core_qa", display=False)
    except Exception as e:
        print("QA report skipped:", e)

    dp_audit = {"enabled": DP["enabled"], "max_epsilon_per_table": DP["max_epsilon"],
                "value_protection_epsilon_per_table": DP["value_protection_epsilon"],
                "epsilon_profiler": PROFILER["epsilon_profiler"], "delta": DP["delta"],
                "n_dp_tables": len(dp_tables), "n_value_protection_tables": len(vp_on),
                "value_protection_off_tables": sorted(vp_off),
                "policy_source": {t: _policies[t]["source"] for t in topo_order(cfg, "core")},
                "train_epsilon_sum": ce["train"], "value_protection_epsilon_sum": ce["valprot"],
                "profiler_epsilon": ce["profiler"], "composed_epsilon_naive_sum": ce["total"],
                "accounting": ("naive_sum: max_epsilon over all DP tables + value_protection_epsilon over "
                               "value-protection-ON tables (resolved by C1 collapse policy) + epsilon_profiler; "
                               "PER-RUN ONLY — cumulative cross-run epsilon is a parked R4 governance item"),
                "rarity_profile_config_hash": RARITY_CONFIG_HASH(), "rarity_profile_fresh": _fresh,
                "model_limits": MODEL_LIMITS, "quick_smoke": QUICK, "gpu_used": bool(gpu_ok),
                "tables": [t["name"] for t in tables]}
    with open(f"{REPORT_DIR}/dp_audit.json", "w") as f:
        json.dump(dp_audit, f, indent=2)
    print("dp_audit:", json.dumps(dp_audit, indent=2))
    return {"composed_epsilon": ce["total"], "value_protection_off": sorted(vp_off),
            "training_status": str(overall)}

In [0]:
# =====================================================================================
# §3 GENERATE — import zips, sample, write RAW gen_* (string keys, INT->Long python
#    objects, completeness gate). Returns {table: rows} for the manifest.
# =====================================================================================
GEN_LOG = f"{REPORT_DIR}/generate.log"
_gen_log_lines = []
def g_log(msg=""):
    line = str(msg)
    _gen_log_lines.append(line)
    print(line)
def g_flush_log():
    try:
        with open(GEN_LOG, "w") as f:
            f.write("\n".join(_gen_log_lines) + "\n")
    except Exception as e:
        print("flush_log failed:", e)

_G_INT = (T.IntegerType, T.LongType, T.ShortType, T.ByteType)
_G_FLT = (T.FloatType, T.DoubleType)

def g_src_schema(name):
    base = GRAPH[name].source_table or name
    return spark.table(f"{SRC}.{base}").schema

def g_key_cols(name):
    spec = GRAPH[name]
    keys = set()
    if spec.pk:
        keys.add(spec.pk)
    if spec.context_fk:
        keys.add(spec.context_fk[0])
    return keys

def g_int_objects(series):
    s = pd.to_numeric(series, errors="coerce")
    return [None if pd.isna(v) else int(round(v)) for v in s]

def g_write_gen(name, pdf):
    if pdf is None or len(pdf) == 0:
        g_log(f"  gen_{name}: EMPTY (not written)")
        return 0
    keys = g_key_cols(name)
    by = {f.name: f.dataType for f in g_src_schema(name).fields}
    fields, data = [], {}
    for c in pdf.columns:
        if c in keys:
            data[c] = [None if pd.isna(x) else str(x) for x in pdf[c]]
            fields.append(T.StructField(c, T.StringType(), True))
            continue
        dt = by.get(c, T.StringType())
        if isinstance(dt, _G_INT):
            data[c] = g_int_objects(pdf[c])
            fields.append(T.StructField(c, T.LongType(), True))
        elif isinstance(dt, _G_FLT):
            s = pd.to_numeric(pdf[c], errors="coerce")
            data[c] = [None if pd.isna(v) else float(v) for v in s]
            fields.append(T.StructField(c, T.DoubleType(), True))
        elif isinstance(dt, (T.DateType, T.TimestampType)):
            s = pd.to_datetime(pdf[c], errors="coerce")
            data[c] = [None if pd.isna(v) else v.to_pydatetime() for v in s]
            fields.append(T.StructField(c, T.TimestampType(), True))
        else:
            data[c] = [None if pd.isna(x) else str(x) for x in pdf[c]]
            fields.append(T.StructField(c, T.StringType(), True))
    rows = list(zip(*[data[f.name] for f in fields])) if fields else []
    sdf = spark.createDataFrame(rows, schema=T.StructType(fields))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{TGT}.gen_{name}")
    nn_keys = {c: int(sum(v is not None for v in data[c])) for c in keys if c in data}
    g_log(f"  gen_{name}: {len(pdf)} rows | key non-null {nn_keys}")
    return len(pdf)


def generate(mostly):
    """Returns {table: rows} across reference + core (the generation record for the manifest)."""
    assert os.path.exists(GEN_REFERENCE), f"missing {GEN_REFERENCE} — reference training required"
    assert os.path.exists(GEN_CORE), f"missing {GEN_CORE} — core training required"
    counts = {}

    g_log("=== reference generation ===")
    gref = mostly.generators.import_from_file(GEN_REFERENCE)
    _ref_root = (cfg.reference_chain or [topo_order(cfg, "reference")[0]])[-1]   # chain ROOT sizes the pool
    sd_ref = mostly.generate(gref, size={_ref_root: ref_pool})
    ref_data = sd_ref.data()
    g_log(f"reference tables returned: {sorted(ref_data)}")
    for name in topo_order(cfg, "reference"):
        if name in ref_data:
            counts[name] = g_write_gen(name, ref_data[name])
        else:
            g_log(f"  gen_{name}: MISSING from generator output")

    g_log("=== core generation ===")
    gcore = mostly.generators.import_from_file(GEN_CORE)
    sd_core = mostly.generate(gcore, size={cfg.subject: output_persons})
    core_data = sd_core.data()
    g_log(f"core tables returned by SDK: {sorted(core_data)}")

    g_log("Core generated row counts (expected vs returned):")
    expected_core = topo_order(cfg, "core")
    for name in expected_core:
        if name in core_data and core_data[name] is not None:
            g_log(f"  {name}: {len(core_data[name])} rows")
        else:
            g_log(f"  {name}: *** MISSING from SDK output ***")

    written = {}
    for name in expected_core:
        if name in core_data:
            written[name] = g_write_gen(name, core_data[name])
            counts[name] = written[name]

    bad = [t for t in expected_core if written.get(t, 0) == 0]
    if bad:
        msg = (f"INCOMPLETE GENERATION — {len(bad)}/{len(expected_core)} core tables missing or "
               f"empty: {bad}. Stale gen_* tables for these were NOT overwritten. Do NOT assemble "
               f"until all core tables are fresh.")
        if allow_incomplete:
            g_log("!!! WARNING (allow_incomplete=true) " + msg)
        else:
            g_log("!!! " + msg)
            g_flush_log()
            raise RuntimeError(msg)

    try:
        sd_core.reports(file_path=f"{REPORT_DIR}/core_data_qa", display=False)
    except Exception as e:
        g_log(f"core data QA skipped: {e}")

    g_log(f"=== DONE: {len(counts)} tables generated, output_persons={output_persons}, empty_core={bad} ===")
    g_flush_log()
    return counts

In [0]:
# =====================================================================================
# DISPATCH — reference (skippable) -> core train (skippable) -> generate (skipped for
# quick_smoke / skip_generate). Records train + generation into the run manifest (R1).
# =====================================================================================
mostly = MostlyAI(local=True, local_dir=LOCAL_DIR)
print("MostlyAI local_dir (effective):", getattr(mostly, "local_dir", "?"))
sdtypes = load_sdtypes()

result = {"run_id": RUN_ID, "skip_reference": SKIP_REFERENCE, "skip_train": SKIP_TRAIN,
          "skip_generate": SKIP_GENERATE, "quick_smoke": QUICK, "gpu_used": bool(gpu_ok)}
train_record = {"recorded_utc": _utcnow(), "quick_smoke": QUICK, "gpu_used": bool(gpu_ok)}

if not SKIP_REFERENCE:
    train_reference(mostly, sdtypes)
    result["reference"] = GEN_REFERENCE
else:
    print(f"skip_reference=true — reusing {GEN_REFERENCE} (exists={os.path.exists(GEN_REFERENCE)})")

if not SKIP_TRAIN:
    core_rec = train_core(mostly, sdtypes)
    result["core_train"] = core_rec
    train_record.update(core_rec)
else:
    print(f"skip_train=true — reusing {GEN_CORE} (exists={os.path.exists(GEN_CORE)})")

train_record["reference_zip"] = _zip_fingerprint(GEN_REFERENCE)
train_record["core_zip"] = _zip_fingerprint(GEN_CORE)

# quick_smoke NEVER generates (the exported model is an under-trained smoke artefact).
do_generate = not SKIP_GENERATE and not QUICK
if QUICK:
    print("quick_smoke — generation DISABLED (restore core.zip.pre_quick_smoke_backup for real use).")
elif SKIP_GENERATE:
    print("skip_generate=true — not generating.")

if do_generate:
    generation_memory_gate()
    counts = generate(mostly)
    result["generate"] = {"n_tables": len(counts), "output_persons": output_persons}
    # R1: record the generation into the run manifest so pipeline stage=post can verify gen_*.
    m = read_manifest() or {"run_id": RUN_ID}
    m["run_id"] = RUN_ID
    m["train"] = train_record
    m["generation"] = {"recorded_utc": _utcnow(), "kind": "mostlyai",
                        "output_persons": output_persons, "ref_pool": ref_pool, "tables": counts}
    m["stage"] = "generated"
    write_manifest(m)
    print(f"run manifest updated: generation recorded ({len(counts)} tables, run_id={RUN_ID})")
else:
    # still record the train section (so a train-only run is traceable)
    m = read_manifest() or {"run_id": RUN_ID}
    m["run_id"] = RUN_ID
    m["train"] = train_record
    if "stage" not in m or m.get("stage") == "cohort_ready":
        m["stage"] = "trained"
    write_manifest(m)

result["status"] = "ok"
print("NEXT: re-run synth_omop/pipeline with stage=post (serverless) for assemble -> validate -> promote.")
dbutils.notebook.exit(json.dumps(result, default=str))